In [1]:
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [3]:
transform= transforms.ToTensor()
training_data = datasets.MNIST(root='data', train=True, download=True, transform=transform)
test_data= datasets.MNIST(root='data', train =False, download=True, transform=transform)

100%|██████████| 9.91M/9.91M [00:00<00:00, 52.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.68MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.4MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.3MB/s]


In [4]:
train_loader= DataLoader(training_data, batch_size=64, shuffle=True)
test_loader=DataLoader(test_data, batch_size=64, shuffle=False)

In [5]:
class MNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.CNN1=nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3)
        self.relu=nn.ReLU()
        self.mxp= nn.MaxPool2d(kernel_size=2, stride=2)
        self.CNN2=nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
        self.flatten=nn.Flatten()
        self.lin1=nn.Linear(1600, 800)
        self.lin2=nn.Linear(800, 10)
    
        
    def forward(self, x):
        x= self.CNN1(x)
        x=self.relu(x)
        x=self.mxp(x)
        x=self.CNN2(x)
        x=self.relu(x)
        x=self.mxp(x)
        x=self.flatten(x)
        x=self.lin1(x)
        x=self.relu(x)
        x=self.lin2(x)
        return x
        

In [6]:
model = MNIST().to(device)
loss_fn = nn.CrossEntropyLoss()
optim = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10
totaloss = 0

for epoch in range(num_epochs):
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)  # 👈 here
        optim.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        totaloss += loss
        loss.backward()
        optim.step()
    print(totaloss.item())
    totaloss = 0

129.87249755859375
40.95191192626953
27.05425453186035
18.956218719482422
15.736515998840332
10.837263107299805
9.069299697875977
7.467859745025635
7.5942792892456055
4.70137882232666


In [7]:
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    print(f'Accuracy: {correct/total*100:.2f}%')

Accuracy: 99.04%


In [9]:
torch.save(model.state_dict(), 'mnist_model.pth')
